# Business Analysis | Retail Customer Churn & Revenue Analysis

This notebook converts cleaned customer records into business questions: where churn is concentrated, how much recurring revenue is exposed, and which customer segments should be prioritized.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data/cleaned/telco_customer_churn_cleaned.csv')
df['churn_flag'] = df['churn'].eq('Yes').astype(int)
df['monthly_revenue_at_risk'] = df['monthly_charges'].where(df['churn_flag'].eq(1), 0)
df.head()

## Executive KPIs

`estimated_clv` is the observed `TotalCharges` value and is used as a transparent customer-value proxy. It is not a predictive lifetime-value model because the public snapshot does not include margin, discount, or future-survival assumptions.

In [ ]:
kpis = pd.Series({
    'Customers': df['customer_id'].nunique(),
    'Churn rate (%)': round(df['churn_flag'].mean() * 100, 2),
    'Monthly recurring revenue': round(df['monthly_charges'].sum(), 2),
    'Monthly revenue at risk': round(df['monthly_revenue_at_risk'].sum(), 2),
    'Avg observed customer value': round(df['estimated_clv'].mean(), 2),
})
kpis

In [ ]:
def segment_rate(column):
    return (df.groupby(column).agg(customers=('customer_id', 'nunique'), churn_rate=('churn_flag', 'mean'), revenue_at_risk=('monthly_revenue_at_risk', 'sum')).assign(churn_rate_pct=lambda x: (x['churn_rate'] * 100).round(2)).drop(columns='churn_rate').sort_values('churn_rate_pct', ascending=False))

contract_summary = segment_rate('contract')
internet_summary = segment_rate('internet_service')
tenure_summary = segment_rate('tenure_band')
contract_summary

## Insight 1: contract risk

Compare month-to-month and two-year contracts to quantify the retention opportunity.

In [ ]:
month_rate = contract_summary.loc['Month-to-month', 'churn_rate_pct']
two_year_rate = contract_summary.loc['Two year', 'churn_rate_pct']
print(f'Month-to-month churn is {month_rate:.1f}% vs {two_year_rate:.1f}% for two-year contracts.')

## Insight 2: early-tenure retention

Because this is a snapshot without a calendar date, retention is shown by tenure band rather than month-over-month cohort trend.

In [ ]:
tenure_summary

In [ ]:
sns.set_theme(style='whitegrid')
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=contract_summary.reset_index(), x='churn_rate_pct', y='contract', hue='contract', legend=False, palette='crest', ax=ax)
ax.set_title('Churn rate by contract type')
ax.set_xlabel('Churn rate (%)')
ax.set_ylabel('')
plt.tight_layout()